# Task: Determine the negative-filtering threshold empirically

## Context

I am building a daily flood occurrence classifier for the Mount Elgon region of
Eastern Uganda (districts: Mbale, Bududa, Sironko, Manafwa, Butaleja,
Kapchorwa, Bulambuli, and optionally Kween and Bukwo). This replicates
Sankaranarayanan et al. (2020), "Flood prediction based on weather parameters
using deep learning", but at daily rather than monthly resolution.

Positive days are extremely rare, roughly 280 flood records against ~80,000
district-days, giving a prevalence near 0.2%. Before training, I need to
restrict the negative class to days where flooding was physically plausible,
so the model learns the boundary between "heavy rain that flooded" and "heavy
rain that did not", rather than the trivial rule that most days are dry.

Your job is to determine that threshold empirically from my data. Do not train
any model. This is a data analysis task that produces a justified number and
the evidence behind it.

## Inputs

- Labelled flood occurrence data: `dataset/flood_labelled_data.csv`
  Expected columns (adapt to what is actually there): `Serial`,`Date (YMD)`,`Day Index`,`Observation Date`,`District`,`Duration (d)`,`Event Days`,`Duration Class`.
  Each row is one recorded flood event for one district.

- Daily rainfall data: `dataset/chirps_daily_rainfall.csv`
  Expected columns: `date,district`,`rain_mean`,`rain_max`,`rain_min`.
  This is catchment-mean daily rainfall in millimetres, extracted per district.

- Output directory for tables and figures: `dataset/`

Inspect both files before assuming anything about their structure, dtypes,
date formats or units. Report what you actually find, including row counts,
date ranges, districts present, and any missing or duplicated records.

## Definitions

- A **positive day** is the onset date of a recorded flood event for that
  district. Use the event start date only. Do not expand events across multiple
  days, even where a duration field exists.
- A **negative day** is any other district-day in the record.
- **Antecedent rainfall** is the rolling cumulative catchment rainfall over a
  window of N days ending on and including the day in question. Compute this
  per district, never pooled across districts, and make sure the rolling window
  does not run across gaps in the date index.

## Steps

1. **Load and join.** Build a complete daily panel of district-days across the
   full overlapping date range of both sources. Mark each day 1 or 0. Report
   the resulting positive count, total row count and prevalence. Flag any flood
   records that could not be matched to rainfall data, and any dates that look
   month-precise rather than day-precise (for example a day value of 1 or 0
   appearing far more often than chance). Exclude unmatched or
   month-precise records from the threshold analysis and report how many.

2. **Compute antecedent rainfall** for windows of 1, 2, 3, 5, 7 and 15 days.

3. **Describe the positive distribution.** For each window, report the minimum,
   1st, 5th, 10th, 25th, 50th and 75th percentiles of antecedent rainfall on
   positive days. Do the same for negative days for comparison.

4. **Choose the discriminating window.** For each window, quantify how well
   antecedent rainfall separates positives from negatives, using AUC of the
   single variable and the ratio of positive median to negative median. Report
   which window separates best. Do not assume 3 days is the answer.

5. **Recommend a threshold.** The rule is: retain at least 99% of positives
   while removing as many negatives as possible. State the recommended value,
   how many positives it excludes, and what prevalence and class ratio result.
   Aim for a class ratio that is trainable, roughly in the range 30:1 to 100:1;
   if no threshold achieves both, say so and explain the trade.

6. **Investigate excluded positives individually.** Any flood day falling below
   the recommended threshold is important. List each one with its date,
   district and antecedent rainfall. These are either label errors, month-precise
   dates, non-rainfall triggers such as blocked drainage or upstream release, or
   cases where the gridded rainfall product missed a local convective storm.
   Do not delete them. Report them for manual review.

## Constraints

- Never filter positive days. The threshold applies only to negatives.
- The rule must depend only on rainfall, which is available at prediction time.
  Do not use any variable derived from the outcome.
- Compute rolling windows within district groups, sorted by date.
- Handle missing rainfall days explicitly. State whether you dropped them,
  treated them as zero, or interpolated, and justify the choice.
- Use whatever units are actually in the file. If rainfall is not in mm,
  convert and say so.

## Outputs

Write to `dataset/`:

1. `threshold_analysis.md` — findings, the recommended threshold with its
   justification, the sensitivity comparison, and a short section listing the
   excluded positives for manual review.
2. A plot of antecedent rainfall distributions for positives against negatives
   at the chosen window, with the recommended threshold marked.
3. The analysis script itself, so the work is reproducible.

Report the numbers you find rather than the numbers you expect. If the data
does not support a clean threshold, say so plainly and explain why.

---

# Analysis

This notebook is the reproducible artefact: running it top to bottom regenerates
`dataset/threshold_analysis.md` and the distribution plot.

**Scope decision.** The rainfall file carries 9 districts; the labels carry 7.
BUKWO and KWEEN were removed from the study area during label preprocessing
(high-massif, landslide-dominated, 1-2 records each), so the panel is built on the
**7 label districts**. Including the other two would reintroduce ~20,000
all-negative district-days for districts with no positives at all.

In [ ]:
import math
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FLOOD_LABELS = "dataset/flood_labelled_data.csv"
RAINFALL = "dataset/chirps_daily_rainfall.csv"
OUTPUT_DIR = "dataset"

RAIN_COLUMN = "rain_mean"          # catchment-mean daily rainfall, mm
WINDOWS = [1, 2, 3, 5, 7, 15]
PERCENTILES = [1, 5, 10, 25, 50, 75]

MIN_POSITIVE_RETENTION = 0.99
RETENTION_LEVELS = [0.99, 0.98, 0.95, 0.90, 0.75, 0.50]
REVIEW_COUNT = 12

# Only the conventional month-precision defaults are tested: testing all 31
# day-of-month values would trip on days with no mechanism behind them
MONTH_PRECISION_CANDIDATES = [1, 15]
MONTH_PRECISION_ALPHA = 0.05

# Validated categorical slots 1 and 2
COLOR_NEG, COLOR_POS = "#2a78d6", "#eb6834"
COLOR_TEXT, COLOR_MUTED, COLOR_SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"


def find_repo_root(marker: str = "dataset") -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(f"No parent of {Path.cwd()} contains {marker!r}")


REPO_ROOT = find_repo_root()
report: list[str] = []


def say(line: str = "") -> None:
    """Print, and record for threshold_analysis.md."""
    print(line)
    report.append(line)


def binomial_tail(observed: int, trials: int, probability: float) -> float:
    """P(X >= observed) for X ~ Binomial(trials, probability), exactly."""
    return sum(
        math.comb(trials, k) * probability**k * (1 - probability) ** (trials - k)
        for k in range(observed, trials + 1)
    )


def auc_from_ranks(positive: np.ndarray, negative: np.ndarray) -> float:
    """Single-variable AUC via the rank identity, with average ranks for ties.

    Ties matter here: rainfall has a large spike at zero, which a tie-naive
    implementation would score optimistically.
    """
    combined = np.concatenate([positive, negative])
    ranks = pd.Series(combined).rank(method="average").to_numpy()
    n_pos, n_neg = len(positive), len(negative)
    return (ranks[:n_pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


print(f"repo root: {REPO_ROOT}")

In [ ]:
labels = pd.read_csv(REPO_ROOT / FLOOD_LABELS, parse_dates=["Observation Date"])
rainfall = pd.read_csv(REPO_ROOT / RAINFALL, parse_dates=["date"])
labels["District"] = labels["District"].str.strip().str.upper()
rainfall["district"] = rainfall["district"].str.strip().str.upper()

say("# Rainfall threshold for negative filtering")
say()
say("Empirically derived threshold on antecedent rainfall, used to restrict the")
say("negative class of a daily flood-occurrence classifier for the Mount Elgon")
say("districts. No model is trained here.")
say()
say("## 1. Data as found")
say()
say(f"**Flood labels** (`{FLOOD_LABELS}`)")
say()
say(f"- {len(labels)} rows, {labels['Serial'].nunique()} events, "
    f"{labels['District'].nunique()} districts")
say(f"- Columns: {', '.join(labels.columns)}")
say(f"- Range: {labels['Observation Date'].min():%Y-%m-%d} to {labels['Observation Date'].max():%Y-%m-%d}")
say(f"- Duplicated district-days: {int(labels.duplicated(['Observation Date', 'District']).sum())}")
say()
say(f"**Rainfall** (`{RAINFALL}`)")
say()
say(f"- {len(rainfall):,} rows, {rainfall['district'].nunique()} districts")
say(f"- Range: {rainfall['date'].min():%Y-%m-%d} to {rainfall['date'].max():%Y-%m-%d}")
say(f"- `{RAIN_COLUMN}`: min {rainfall[RAIN_COLUMN].min():.2f}, max {rainfall[RAIN_COLUMN].max():.2f}, "
    f"mean {rainfall[RAIN_COLUMN].mean():.2f} mm/day")
say(f"- Nulls: {int(rainfall[RAIN_COLUMN].isna().sum())}; negative values: "
    f"{int((rainfall[RAIN_COLUMN] < 0).sum())}; duplicated district-days: "
    f"{int(rainfall.duplicated(['date', 'district']).sum())}")
say()

STUDY_DISTRICTS = sorted(labels["District"].unique())
dropped_districts = sorted(set(rainfall["district"].unique()) - set(STUDY_DISTRICTS))
rainfall = rainfall[rainfall["district"].isin(STUDY_DISTRICTS)]

say(f"The rainfall file covers {len(STUDY_DISTRICTS) + len(dropped_districts)} districts and the "
    f"labels {len(STUDY_DISTRICTS)}. **{', '.join(dropped_districts)} are excluded** — they were "
    "removed from the study area during label preprocessing as high-massif, landslide-dominated "
    "districts with 1-2 records each. Keeping their rainfall would add districts that can only "
    "ever contribute negatives.")
say()
say(f"Study area: {', '.join(STUDY_DISTRICTS)}")
say()

In [ ]:
onsets = labels[labels["Day Index"] == 1].copy()

# Month-precision screen on DISTINCT dates. Onset rows are not independent - a
# multi-district event contributes one row per district on a single date - so
# testing rows would read district multiplicity as date clustering.
distinct = onsets["Observation Date"].drop_duplicates()
counts = distinct.dt.day.value_counts()
expected = len(distinct) / 30.44
alpha = MONTH_PRECISION_ALPHA / len(MONTH_PRECISION_CANDIDATES)

screen = []
for day in MONTH_PRECISION_CANDIDATES:
    observed = int(counts.get(day, 0))
    p_value = binomial_tail(observed, len(distinct), 1 / 30.44)
    screen.append((day, observed, p_value, observed > 2 * expected and p_value < alpha))

suspect_days = [d for d, _, _, flagged in screen if flagged]
month_precise = onsets[onsets["Observation Date"].dt.day.isin(suspect_days)]
clean = onsets[~onsets["Observation Date"].dt.day.isin(suspect_days)]

say("### Positives and the date screens")
say()
say(f"A positive is an event **onset** — `Day Index == 1` — giving {len(onsets)} candidate "
    f"positives from {len(labels)} labelled event-days. Later days of an event are not positives: "
    "their antecedent rainfall reflects a flood already in progress.")
say()
say(f"Month-precision screen, on the {len(distinct)} distinct onset dates rather than the "
    f"{len(onsets)} rows (expected {expected:.2f} dates per day-of-month, alpha {alpha:.3f}):")
say()
for day, observed, p_value, flagged in screen:
    say(f"- day {day}: {observed} distinct dates, p = {p_value:.4f} — "
        f"{'**flagged, excluded**' if flagged else 'not significant, retained'}")
say()
if len(month_precise):
    say(f"{len(month_precise)} onsets fall on a flagged day. Their district-days are dropped from "
        "the panel rather than counted as negatives: if the date is month-precise the flood still "
        "happened that month, so the day is unknown rather than flood-free.")
say()

start = max(rainfall["date"].min(), onsets["Observation Date"].min())
end = min(rainfall["date"].max(), onsets["Observation Date"].max())

index = pd.MultiIndex.from_product(
    [pd.date_range(start, end, freq="D"), STUDY_DISTRICTS], names=["date", "district"]
)
panel = pd.DataFrame(index=index).reset_index().merge(rainfall, on=["date", "district"], how="left")

in_range = clean[clean["Observation Date"].between(start, end)]
unmatched = clean[~clean.index.isin(in_range.index)]

positive_keys = set(zip(in_range["Observation Date"], in_range["District"]))
panel["flood"] = [int((d, x) in positive_keys) for d, x in zip(panel["date"], panel["district"])]

suspect_keys = set(zip(month_precise["Observation Date"], month_precise["District"]))
suspect = np.array([(d, x) in suspect_keys for d, x in zip(panel["date"], panel["district"])])
panel = panel[~suspect].reset_index(drop=True)

say("### Panel")
say()
say(f"- Overlapping range: {start:%Y-%m-%d} to {end:%Y-%m-%d}")
say(f"- District-days: {len(panel):,} across {len(STUDY_DISTRICTS)} districts")
say(f"- Positive days: {int(panel['flood'].sum())}")
say(f"- Prevalence: **{100 * panel['flood'].mean():.3f}%**")
say(f"- Missing rainfall inside the panel: {int(panel[RAIN_COLUMN].isna().sum())} — the rainfall "
    "record is a complete daily panel, so nothing is dropped, zero-filled or interpolated")
say(f"- District-days removed as month-precise: {int(suspect.sum())}")
say()
if len(unmatched):
    say(f"**{len(unmatched)} onsets predate the rainfall record and are excluded:** "
        + ", ".join(f"{r['Observation Date']:%Y-%m-%d} {r['District']}" for _, r in unmatched.iterrows()))
    say()

In [ ]:
panel = panel.sort_values(["district", "date"]).reset_index(drop=True)
grouped = panel.groupby("district")[RAIN_COLUMN]
for window in WINDOWS:
    # min_periods=window leaves the first N-1 days null rather than crediting a
    # short sum; the panel is gap-free so no window straddles a break
    panel[f"ante_{window}d"] = grouped.transform(
        lambda s, w=window: s.rolling(w, min_periods=w).sum()
    )

say("## 2. Antecedent rainfall distributions")
say()
say(f"Rolling sums over {WINDOWS} days, computed within district groups sorted by date. "
    "Percentiles in mm:")
say()
say("| Window | Class | n | min | " + " | ".join(f"p{p}" for p in PERCENTILES) + " |")
say("| " + " --- |" * (4 + len(PERCENTILES)))
for window in WINDOWS:
    usable = panel.dropna(subset=[f"ante_{window}d"])
    for label, mask in (("positive", usable["flood"] == 1), ("negative", usable["flood"] == 0)):
        values = usable.loc[mask, f"ante_{window}d"].to_numpy()
        cells_ = " | ".join(f"{np.percentile(values, p):.1f}" for p in PERCENTILES)
        say(f"| {window}d | {label} | {len(values):,} | {values.min():.1f} | {cells_} |")
say()

In [ ]:
rows = []
for window in WINDOWS:
    usable = panel.dropna(subset=[f"ante_{window}d"])
    positive = usable.loc[usable["flood"] == 1, f"ante_{window}d"].to_numpy()
    negative = usable.loc[usable["flood"] == 0, f"ante_{window}d"].to_numpy()
    pos_med, neg_med = float(np.median(positive)), float(np.median(negative))
    rows.append({
        "window_days": window,
        "auc": auc_from_ranks(positive, negative),
        "positive_median": pos_med,
        "negative_median": neg_med,
        "median_ratio": pos_med / neg_med if neg_med > 0 else float("inf"),
    })
separation = pd.DataFrame(rows)

say("## 3. Which window separates best")
say()
say("| Window | AUC | Positive median | Negative median | Median ratio |")
say("| --- | --- | --- | --- | --- |")
for _, r in separation.iterrows():
    say(f"| {int(r['window_days'])}d | {r['auc']:.4f} | {r['positive_median']:.1f} | "
        f"{r['negative_median']:.1f} | {r['median_ratio']:.2f} |")
say()
best = separation.loc[separation["auc"].idxmax()]
WINDOW = int(best["window_days"])
say(f"**Best window: {WINDOW} days**, AUC {best['auc']:.4f}. Selected on AUC, not assumed — "
    "3 days is not the answer here.")
say()
say(f"Every window sits between {separation['auc'].min():.2f} and {separation['auc'].max():.2f}, "
    "so antecedent rainfall is a **weak single discriminator at any horizon**. The median ratio "
    "moves the other way to AUC — it is highest at 1 day and lowest at 15 — because a long "
    "window raises both classes together.")
say()

In [ ]:
COLUMN = f"ante_{WINDOW}d"
usable = panel.dropna(subset=[COLUMN])
positive = np.sort(usable.loc[usable["flood"] == 1, COLUMN].to_numpy())
negative = usable.loc[usable["flood"] == 0, COLUMN].to_numpy()


def at_retention(retention: float) -> dict:
    """Largest threshold retaining at least `retention` of positives.

    Read off the sorted positives rather than a sweep grid, so the answer is not
    limited by step size.
    """
    keep = int(math.ceil(retention * len(positive)))
    threshold = float(positive[len(positive) - keep])
    kept_p = int((positive >= threshold).sum())
    kept_n = int((negative >= threshold).sum())
    return {
        "retention_pct": 100 * retention,
        "threshold_mm": threshold,
        "positives_kept": kept_p,
        "positives_dropped": len(positive) - kept_p,
        "negatives_removed": len(negative) - kept_n,
        "negatives_removed_pct": 100 * (len(negative) - kept_n) / len(negative),
        "prevalence_pct": 100 * kept_p / (kept_p + kept_n),
        "class_ratio": kept_n / kept_p,
    }


trade = pd.DataFrame([at_retention(r) for r in RETENTION_LEVELS])
chosen = trade.iloc[0]
THRESHOLD = float(chosen["threshold_mm"])

say("## 4. Recommendation")
say()
say(f"**Threshold: {THRESHOLD:.1f} mm of {WINDOW}-day antecedent rainfall**, applied to "
    "negatives only. Positives are never filtered.")
say()
say(f"- Positives retained: {int(chosen['positives_kept'])} of {len(positive)} "
    f"({100 * chosen['positives_kept'] / len(positive):.1f}%)")
say(f"- Negatives removed: {int(chosen['negatives_removed']):,} of {len(negative):,} "
    f"({chosen['negatives_removed_pct']:.1f}%)")
say(f"- Prevalence: {100 * panel['flood'].mean():.3f}% -> {chosen['prevalence_pct']:.3f}%")
say(f"- Class ratio: **{chosen['class_ratio']:.0f}:1**")
say()
say("### The data does not support a clean threshold")
say()
say("Stated plainly: the rule as specified — retain 99% of positives, reach a 30:1-100:1 class "
    f"ratio — **cannot be satisfied by any rainfall threshold on this data.** At 99% retention the "
    f"ratio is {chosen['class_ratio']:.0f}:1, roughly six times the top of the target band.")
say()
say("Two independent causes:")
say()
say(f"1. **Prevalence is far below the brief's estimate.** The task assumed ~280 flood records "
    f"against ~80,000 district-days (0.2%). After reducing to onsets, restricting to the 7-district "
    f"study area and excluding pre-1998 and month-precise dates, there are **{len(positive)} "
    f"positives against {len(negative):,} negatives** — {100 * panel['flood'].mean():.3f}%. "
    "Reaching 100:1 from ~590:1 needs about 85% of negatives removed.")
say(f"2. **Some flood days carry almost no antecedent rainfall.** The two lowest positives sit at "
    f"{positive[0]:.1f} and {positive[1]:.1f} mm over {WINDOW} days, against a negative 5th "
    f"percentile of {np.percentile(negative, 5):.1f} mm. A rule that must keep 99% of positives "
    "cannot raise the threshold past them.")
say()
say(f"What relaxing the retention rule would buy, on the {WINDOW}-day window:")
say()
say("| Positive retention | Threshold (mm) | Positives dropped | Negatives removed | Ratio |")
say("| --- | --- | --- | --- | --- |")
for _, r in trade.iterrows():
    say(f"| {r['retention_pct']:.0f}% | {r['threshold_mm']:.1f} | {int(r['positives_dropped'])} | "
        f"{int(r['negatives_removed']):,} ({r['negatives_removed_pct']:.0f}%) | "
        f"{r['class_ratio']:.0f}:1 |")
say()
reachable = trade[trade["class_ratio"] <= 100]
if len(reachable):
    r = reachable.iloc[0]
    say(f"Only at {r['retention_pct']:.0f}% retention does the ratio reach "
        f"{r['class_ratio']:.0f}:1 — at the cost of {int(r['positives_dropped'])} flood days.")
else:
    worst = trade.iloc[-1]
    say(f"**No retention level reaches 100:1.** Even discarding "
        f"{int(worst['positives_dropped'])} of {len(positive)} positives — half the flood "
        f"record — leaves {worst['class_ratio']:.0f}:1.")
say()
say("### What to do instead")
say()
say(f"Keep the {THRESHOLD:.1f} mm filter. It costs no positives, removes "
    f"{int(chosen['negatives_removed']):,} district-days on which a flood was not physically "
    "plausible, and is defensible: a day with almost no rain in the preceding fortnight is not a "
    "case the model should be asked to rule out. But it does not solve the imbalance, and should "
    "not be presented as if it does.")
say()
say("The imbalance needs a different instrument:")
say()
say("- **Class weights or focal loss** sized to the true ratio. Standard for rare-event "
    "classification, and discards nothing.")
say("- **Negative subsampling** to a chosen ratio, stratified by season and district, keeping "
    "every positive — described as training-set construction, not data cleaning.")
say("- **Threshold-free evaluation**: precision-recall AUC and the PR curve. At 0.16% prevalence "
    "accuracy and ROC-AUC are both misleading.")
say()

In [ ]:
review = usable[usable["flood"] == 1].nsmallest(REVIEW_COUNT, COLUMN)[
    ["date", "district", COLUMN, RAIN_COLUMN]
].copy()
review.columns = ["date", "district", f"antecedent_{WINDOW}d_mm", "same_day_mm"]
review["dropped_at_retention"] = [
    next((f"{r['retention_pct']:.0f}%" for _, r in trade.iterrows() if v < r["threshold_mm"]),
         "retained at all levels")
    for v in review[f"antecedent_{WINDOW}d_mm"]
]

excluded_now = int((positive < THRESHOLD).sum())

say("## 5. Positives at the low tail, for manual review")
say()
say(f"{excluded_now} flood day(s) fall below the recommended {THRESHOLD:.1f} mm threshold. That is "
    f"too few to review usefully, so the table lists the {len(review)} onsets with the lowest "
    f"{WINDOW}-day antecedent rainfall — the records that pin the threshold down, and the first a "
    "stricter rule would discard. **None are deleted.**")
say()
say("Each is likely a label error, a date less precise than it looks, a non-rainfall trigger such "
    "as blocked drainage or an upstream release, or a local convective storm the gridded product "
    "smoothed away. A flood onset preceded by under 10 mm over a fortnight is not plausibly "
    "rainfall-driven.")
say()
say(f"| Date | District | {WINDOW}-day antecedent (mm) | Same-day (mm) | Dropped at |")
say("| --- | --- | --- | --- | --- |")
for _, r in review.iterrows():
    say(f"| {r['date']:%Y-%m-%d} | {r['district']} | {r[f'antecedent_{WINDOW}d_mm']:.1f} | "
        f"{r['same_day_mm']:.2f} | {r['dropped_at_retention']} |")
say()
say("The top two are the binding constraint on the whole analysis. Both are MBALE, both in March, "
    "and both record under 7 mm across the preceding fortnight — worth checking against the "
    "original DesInventar comments before they are trusted as flood days.")
say()

say("## 6. Sensitivity")
say()
say("The recommendation against neighbouring thresholds:")
say()
say("| Threshold (mm) | Positives kept | Negatives removed | Prevalence % | Ratio | |")
say("| --- | --- | --- | --- | --- | --- |")
for value in [0.0, THRESHOLD / 2, THRESHOLD, THRESHOLD * 2, THRESHOLD * 4]:
    kept_p = int((positive >= value).sum())
    kept_n = int((negative >= value).sum())
    marker = " **recommended**" if abs(value - THRESHOLD) < 1e-9 else ""
    say(f"| {value:.1f} | {kept_p}/{len(positive)} | {len(negative) - kept_n:,} "
        f"({100 * (len(negative) - kept_n) / len(negative):.1f}%) | "
        f"{100 * kept_p / (kept_p + kept_n):.3f} | {kept_n / kept_p:.0f}:1 |{marker} |")
say()
say(f"Nothing turns on the exact value. Quadrupling the threshold to {THRESHOLD * 4:.1f} mm "
    "removes only a further 9% of negatives and still leaves the ratio above 500:1, while "
    "costing 3 positives. No setting in this range changes the conclusion — which is another "
    "way of saying rainfall is not doing much work here.")
say()

In [ ]:
figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4.8), facecolor=COLOR_SURFACE)
upper = float(np.percentile(negative, 99.9))

for axis in (left, right):
    axis.set_facecolor(COLOR_SURFACE)
    axis.grid(True, color="#e6e5e1", linewidth=0.8)
    axis.set_axisbelow(True)
    for spine in ("top", "right"):
        axis.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        axis.spines[spine].set_color("#d4d3ce")
    axis.tick_params(colors=COLOR_MUTED, labelsize=9)

# ECDF leads: it answers the question the threshold rule asks - what share of each
# class sits below a given millimetre value
for values, colour, label in ((negative, COLOR_NEG, "Non-flood days"),
                              (positive, COLOR_POS, "Flood onset days")):
    ordered = np.sort(values)
    left.plot(ordered, np.arange(1, len(ordered) + 1) / len(ordered) * 100,
              color=colour, linewidth=2, label=label)
left.axvline(THRESHOLD, color=COLOR_TEXT, linewidth=1.4, linestyle="--")
left.annotate(f"threshold {THRESHOLD:.1f} mm", xy=(THRESHOLD, 55), xytext=(7, 0),
              textcoords="offset points", fontsize=9, color=COLOR_TEXT, rotation=90, va="center")
left.set_xlim(0, upper); left.set_ylim(0, 100)
left.set_xlabel(f"{WINDOW}-day antecedent rainfall (mm)", fontsize=10, color=COLOR_MUTED)
left.set_ylabel("Cumulative share of days (%)", fontsize=10, color=COLOR_MUTED)
left.set_title("Cumulative distribution", fontsize=11, color=COLOR_TEXT, loc="left", pad=10)
left.legend(frameon=False, fontsize=9, labelcolor=COLOR_MUTED, loc="lower right")

# Bin count set by the smaller class (~83 values); finer bins turn it into noise
bins = np.linspace(0, upper, 20)
for values, colour, label in ((negative, COLOR_NEG, "Non-flood days"),
                              (positive, COLOR_POS, "Flood onset days")):
    right.hist(values, bins=bins, density=True, histtype="step",
               linewidth=2, color=colour, label=label)
right.axvline(THRESHOLD, color=COLOR_TEXT, linewidth=1.4, linestyle="--")
right.set_xlim(0, upper)
right.set_xlabel(f"{WINDOW}-day antecedent rainfall (mm)", fontsize=10, color=COLOR_MUTED)
right.set_ylabel("Density", fontsize=10, color=COLOR_MUTED)
right.set_title("Distribution shape", fontsize=11, color=COLOR_TEXT, loc="left", pad=10)
right.legend(frameon=False, fontsize=9, labelcolor=COLOR_MUTED)

figure.suptitle(
    f"Antecedent rainfall on flood onset days vs non-flood days ({WINDOW}-day window)",
    fontsize=13, color=COLOR_TEXT, x=0.008, ha="left", y=0.99,
)
figure.tight_layout(rect=(0, 0, 1, 0.95))

PLOT_PATH = REPO_ROOT / OUTPUT_DIR / "antecedent_rainfall_distribution.png"
figure.savefig(PLOT_PATH, dpi=150, facecolor=COLOR_SURFACE)
plt.close(figure)

say("## Outputs")
say()
say("| File | Contents |")
say("| --- | --- |")
say("| `threshold_analysis.md` | This report |")
say("| `antecedent_rainfall_distribution.png` | Distribution plot with the threshold marked |")
say("| `src/scripts/empirical_rainfall_threshold.ipynb` | The notebook that produced both |")

REPORT_PATH = REPO_ROOT / OUTPUT_DIR / "threshold_analysis.md"
REPORT_PATH.write_text("\n".join(report) + "\n")
print(f"\nwrote {REPORT_PATH}")
print(f"wrote {PLOT_PATH}")

## Outcome

**Recommended: 5.3 mm of 15-day antecedent rainfall**, negatives only. Removes
3,616 district-days (6.9%) at zero cost in positives.

**The specified rule cannot be met.** Retaining 99% of positives while reaching a
30:1–100:1 class ratio is impossible on this data — the ratio stays near 585:1,
and even discarding half the flood record only reaches 252:1.

Two causes, both independent of the threshold choice:

- **83 positives against 52,143 negatives (0.159%)**, well below the 0.2% the brief
  assumed. Reaching 100:1 needs ~85% of negatives gone.
- **Two flood days record 5.3 and 6.5 mm over 15 days**, against a negative 5th
  percentile of 2.8 mm — they sit inside the dry tail of the negative class and pin
  the threshold there.

Antecedent rainfall is a weak discriminator at every horizon (AUC 0.66–0.71). That
is itself the useful finding: rainfall alone will not carry this classifier, which
is consistent with Sankaranarayanan et al. (2020) using multiple weather
parameters. Soil moisture and terrain are likely to matter more than a longer
rainfall window.

Handle the imbalance with class weights or deliberate negative subsampling, and
evaluate on precision-recall rather than ROC-AUC.